In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 

import hyperspy.api as hs
import lumispy as lum

# Some environments (like the linked Binder) cannot support the Qt interface, so it may not work properly in those cases.
# Consider using alternative backends like %matplotlib inline or %matplotlib notebook for compatibility in such environments.
# However, note that some of the presented interactive functionalities will not work with `inline` plotting.
%matplotlib qt

# Batch spectra processing

#### Loading files


HyperSpy supports a wide range of microscopy and spectroscopy related **data file types**:
- Horiba JobinYvon XML `.xml`
- TriVista XML `.tvf`
- Renishaw wire format `.wdf`
- Gatan `.dm3/.dm4`
- Attolight `.sur` (For the moment does not parse `original_metadata` to `metadata`)
- Delmic `.hdf5` 
- Hamamatsu streak camera images in `.tif` format (Reading the itex `.img` format will be available in RosettaSciIO)

For saving analyses, HyperSpy has its own hdf5-based data format `.hspy`.

*We assume the file location as in the demo repository, if you downloaded the notebook and the data files individually, you might need to adapt the path.*

We first load multiple cathodoluminescence spectra from a folder, and *stack* them on top of one another, loading them as one signal that we can process.

In [ ]:
spectra = hs.load('data/CL_spec*.hspy', stack = True) 

In [ ]:
hs.plot.plot_spectra(spectra)

We can inspect the different axes of our signal by calling the `axes_manager`. For this data our spectra have a signal axis `Wavelength`, and a navigational axis where our spectra have been stacked.

In [ ]:
spectra.axes_manager

#### Remove background (interactive)

HyperSpy has an interactive tool for **background removal** that supports various functions, let's start by removing a **simple offset**:

1. Select a region to be used to determine the background: On the signal plot click, drag and release
2. You can still move the region or its boundaries with the mouse and inspect the different spectra using the navigator to make sure the region is right
3. Press `Apply`


In [ ]:
spectra.remove_background(background_type="Offset")

#### Jacobian transformation

To preserve the integrated intensity per spectral window, a *Jacobian* transformation has to be applied to the signal intensity. As we require $I(E)dE = I(\lambda)d\lambda$, the scale transformation $E=hc/\lambda$ implies:

$$I(E) = I(\lambda)\frac{d\lambda}{dE} = I(\lambda)\frac{d}{dE}
\frac{h c}{E} = - I(\lambda) \frac{h c}{E^2}$$

To visualize the effect of the *Jacobian transformation*, we can plot a signal with constant intensity before and after the transformation:

![Alt Text](graphics/jacobian_transformation.png)

This transformation is the default in LumiSpy, but can be deactivated with `jacobian=False`. Converting between $nm$, $eV$ and $cm^{-1}$ can be easily done with wrapper functions in LumiSpy, in this case we convert to $eV$.

In [ ]:
spectra.to_eV(inplace = True)
spectra.axes_manager

The wavelength to energy conversion leads to spectral windows of `non-uniform` widths, in order to complete the next analysis steps we can interpolate along this energy axis and replace the axis with a `uniform` one.

In [ ]:
spectra.interpolate_on_axis('uniform', 
                            axis='Energy', 
                            inplace = True)

#### Peak analysis
HyperSpy has several peak analysis wrapper functions, such as `.find_peaks_ohaver`... For our data we compute the full width half maxima (FWHM) of our spectra.

In [ ]:
fwhms = spectra.estimate_peak_width()
fwhms.data

We can also easily obtain the centroid in the energy axis of each spectrum in our signal.

In [ ]:
centroids = spectra.centroid()
centroids.data

#### Plotting results
Next we can inspect our processed data, and also add our FWHM and centroid values to the legend.

In [ ]:
hs.plot.plot_spectra(spectra, 
                     legend=[
        f"{c:.2f} eV, {f*1000:.3g} meV"
        for c, f in zip(centroids.data, fwhms.data)
    ])

Finally, we can save the changes we've made to our spectra as one file. Alternatively, the seperated spectra can be acessed and save individually with the `spectra.split()` .

In [ ]:
spectra.save('spectra_stack_scaled_eV.hspy')

# Spectral map *Hyperspectral* analysis

#### Load and explore data
*(In the following, we will use the dataset `spectra_map` saved in the Gatan format. The sample contains (In,Ga)N measured by Aidan Campbell at the Paul Drude Institute, Berlin)*.

In [ ]:
spectra_map = hs.load('data/InGaN_hyperspectral.dm4').inav[13:,13:] # (optionally) reduce the spatial extent of the data that we're working with

Our sample dataset has **two navigation dimensions** (47, 47 and **one signal (spectral) dimension** |1336). When we plot the data as it is, we can navigate the map across the 47, 47 spatial dimensions.

In [ ]:
spectra_map.plot()

#### Clean cosmic rays from spectra
`.spikes_removal_tool` can automatically clean your data or can be used interactively where removal of each spike is confirmed by user 

In [ ]:
spectra_map.spikes_removal_tool(interactive = False) 

#### Chromatic imaging

Indexing can also be used for color-filtered (chromatic) imaging. First, lets plot the map with its intensity summed across the wavelength axis:

*(the object is transposed, so that we plot the intensity over navigation instead of signal dimensions)*

In [ ]:
summed_map = spectra_map.T.mean()
summed_map.plot(cmap='inferno')

Now, we can **plot the intensity at a selected spectral value or window** using indexing for intensity at λ=500 nm.

In [ ]:
spectra_map.isig[500.0].plot()

Alternatively, we can explore spectral map interactively by integrating over an adaptable region of interest (ROI).

In [ ]:
rois, [map1, map2] = hs.plot.plot_roi_map(spectra_map, rois = 2, 
                     single_figure = True)

We can directly access the ROIs and the new maps we've extracted.

In [ ]:
rois, [map1, map2]

#### Combined plot
Next we combine our results into one large combined interactive figure, we define Rectangular ROIs for this example, though Point, Circular, Line and Polygon ones are also available.

In [ ]:
# Declare matplotlib figure
fig = plt.figure(figsize = [7,6])
subfigs = fig.subfigures(2, 2, wspace=0.07)

# Declare our ROIs
roi1 = hs.roi.RectangularROI()
roi2 = hs.roi.RectangularROI()

# Plot subplots
spectra_map.plot(colorbar=False)
summed_map.plot(colorbar=False, fig = subfigs[0,0], cmap = 'inferno', title = 'Summed')
map1.plot(cmap = 'Oranges', fig=subfigs[1,0], colorbar=False, title= f"{rois[0][0]:.3g} - {rois[0][1]:.3g} nm")
map2.plot(cmap = 'Blues', fig=subfigs[1,1], colorbar=False, title= f"{rois[1][0]:.3g} - {rois[1][1]:.3g} nm")

# Attach ROIs to relevant plots
roi1.add_widget(spectra_map)
roi1.interactive(map2)
roi1.interactive(summed_map)

roi2.add_widget(spectra_map)
roi2.interactive(map1)
roi2.interactive(summed_map)

# Compute mean signal inside of ROIs
def roi_mean_func(s, roi):
    return roi(s).mean()

roi1_spectra_mean = hs.interactive(roi_mean_func, 
                                event=roi1.events.changed,
                                s = spectra_map,
                                roi = roi1)
roi2_spectra_mean = hs.interactive(roi_mean_func, 
                                event=roi2.events.changed,
                                s = spectra_map,
                                roi = roi2)

# Plot extracted mean signals
hs.plot.plot_spectra([roi1_spectra_mean, roi2_spectra_mean, spectra_map.mean()],
                     legend = ['ROI 1','ROI 2','Mean'],
                     colors = ['blue','orange','black'],
                     fig=subfigs[0,1],
                     normalise = True)
plt.tight_layout()

# Time-resolved luminescence analysis

*(In the following, we will use the preprocessed dataset `streak_image` saved in the HyperSpy format. The sample contains an AlN layer measured by Domenik Spallek at the Paul Drude Insitute, Berlin.)*

The streak images are of the `LumiTransientSpectrum` signal class:

In [ ]:
streak_image = hs.load('data/AlN_HR_streak_image.hspy')
streak_image

We can interactively inspect our streak image and compare transients from the `sum` of vertical slices on the image using two ROIs.

In [ ]:
# Combine plots in one figure
fig = plt.figure(figsize = [8,5])
subfigs = fig.subfigures(2, 2, wspace=0.07, height_ratios=[1, 3])

# Declare ROIs
roi1 = hs.roi.RectangularROI()
roi2 = hs.roi.RectangularROI()

# Plot streak image
streak_image.plot(fig=subfigs[1,0], colorbar = False, 
              title = 'Streak image', cmap = 'turbo')

# Connect ROIs to plot
roi1.add_widget(streak_image, color="red")
roi2.add_widget(streak_image , color="blue")

# Declare function for meaning along wavelength axis
def roi_mean_function(s, roi):
    return roi(s).mean(axis='Wavelength')

# Compute function and interactivity
s_roi_mean1 = hs.interactive(
    roi_mean_function,
    event=roi1.events.changed,
    s=streak_image,
    roi=roi1,
)

s_roi_mean2 = hs.interactive(
    roi_mean_function,
    event=roi2.events.changed,
    s=streak_image,
    roi=roi2,
)
    
# Plot extracted transients and summed spectrum
hs.plot.plot_spectra([s_roi_mean1,
                      s_roi_mean2], 
                     color = ['red','blue'], 
                     normalise = True,
                     fig=subfigs[1,1])

hs.plot.plot_spectra([streak_image.sum(axis = 'Time')], 
                     color = ['k'], 
                     fig=subfigs[0,0])

When calling the signal interactively extracted from the streak image, we can see the new signal type is assigned appropriately.

In [ ]:
s_roi_mean1